In [ ]:
import pandas as pd
import numpy as np

interaction = pd.read_parquet(
    "../data/processed/interaction_matrix.parquet"
)

interaction.shape

In [ ]:
interaction.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [ ]:
similarity_matrix = cosine_similarity(
    interaction
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=interaction.index,
    columns=interaction.index
)

similarity_df.shape

In [ ]:
sample_customer = interaction.index[0]

similar_users = (
    similarity_df[sample_customer]
    .sort_values(
        ascending=False
    )
)

similar_users.head(10)

In [ ]:
def recommend_products(
    customer_id,
    top_n=5
):
    
    similar_users = (
        similarity_df[customer_id]
        .sort_values(
            ascending=False
        )[1:11]
    )

    customer_products = set(
        interaction.loc[
            customer_id
        ][
            interaction.loc[
                customer_id
            ] > 0
        ].index
    )

    recommendations = {}

    for user in similar_users.index:

        similarity_score = (
            similar_users[user]
        )

        purchased = (
            interaction.loc[user]
        )

        for sku in purchased[
            purchased > 0
        ].index:

            if sku not in customer_products:

                recommendations[
                    sku
                ] = (
                    recommendations.get(
                        sku,
                        0
                    )
                    +
                    similarity_score
                )

    recommendations = (
        sorted(
            recommendations.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    return recommendations[:top_n]

In [ ]:
sample_customer = (
    interaction.index[0]
)

recommend_products(
    sample_customer
)

In [ ]:
sku_lookup = pd.read_parquet(
    "../data/processed/sku_lookup.parquet"
)

sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

In [ ]:
sample_customer = (
    interaction.index[0]
)

recs = recommend_products(
    sample_customer
)

for sku, score in recs:

    print(
        sku_dict.get(
            sku,
            sku
        ),
        round(score, 2)
    )

In [ ]:
recommend_products(
    interaction.index[0]
)

In [ ]:
for customer in interaction.index[:5]:

    print("\n")
    print("Customer:", customer)

    recs = recommend_products(
        customer
    )

    for sku, score in recs:

        print(
            sku_dict.get(
                sku,
                sku
            )
        )

In [ ]:
similarity_df.to_parquet(
    "../data/processed/similarity_matrix.parquet"
)

print("Saved")

In [ ]:
interaction.shape

In [ ]:
similarity_df.to_parquet(
    "../data/processed/similarity_matrix.parquet"
)

print("Similarity Matrix Saved")

In [ ]:
similarity_df.shape